<a href="https://colab.research.google.com/github/AmanuelDaget/GBPUSD-Foreign-Exchange-Price-Pridiction-Using-LSTM-GRU-BiLSTM/blob/main/GBPUSD_Price_pridiction_Using_LSTM_GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Libraries**

In [1]:
import argparse, warnings, os
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")


**Config**

In [3]:
DATA_PATH = "/content/data/gbpusd_hourly_newyorkTimezone.csv"
TARGET     = "close"
WINDOW     = 60
UNITS      = 64
DROPOUT    = 0.2
BATCH      = 32
EPOCHS     = 100
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
os.makedirs("results", exist_ok=True)

**Load Data**

In [4]:
df = pd.read_csv(DATA_PATH, skiprows=[0, 1])
df.columns = ['Datetime', 'Open', 'High', 'Low', 'Close']
df['Datetime'] = pd.to_datetime(df['Datetime'], utc=True)
df = df.sort_values('Datetime').set_index('Datetime')
df = df[['Open', 'High', 'Low', 'Close']].astype(float).ffill().dropna()
print(f"Loaded {len(df):,} rows  {df.index[0].date()} → {df.index[-1].date()}")

Loaded 11,825 rows  2024-06-16 → 2026-05-15


**Features**

In [5]:
c = df["Close"]
df["Ret"]     = c.pct_change()
df["LogRet"]  = np.log(c / c.shift(1))
for w in [5, 10, 20]:   df[f"SMA{w}"] = c.rolling(w).mean()
for w in [10, 20, 50]:  df[f"EMA{w}"] = c.ewm(span=w, adjust=False).mean()
d = c.diff()
df["RSI"]     = 100 - 100 / (1 + d.clip(0).rolling(14).mean() /
                              (-d.clip(upper=0)).rolling(14).mean().replace(0, 1e-9))
df["MACD"]    = c.ewm(12, adjust=False).mean() - c.ewm(26, adjust=False).mean()
df["MACDsig"] = df["MACD"].ewm(9, adjust=False).mean()
mid = c.rolling(20).mean(); std = c.rolling(20).std()
df["BBw"]     = (4 * std) / (mid + 1e-9)
tr = pd.concat([df.High - df.Low,
                (df.High - c.shift()).abs(),
                (df.Low  - c.shift()).abs()], axis=1).max(axis=1)
df["ATR"]     = tr.rolling(14).mean()
df["HLr"]     = df.High - df.Low
df["Body"]    = (df.Close - df.Open).abs()
df.dropna(inplace=True)
cols = list(df.columns)
close_idx = cols.index("Close")
print(f"Features: {cols}")

Features: ['Open', 'High', 'Low', 'Close', 'Ret', 'LogRet', 'SMA5', 'SMA10', 'SMA20', 'EMA10', 'EMA20', 'EMA50', 'RSI', 'MACD', 'MACDsig', 'BBw', 'ATR', 'HLr', 'Body']


# **Preprocess**

**Handle Missing value**

In [8]:
missing_before = df.isnull().sum()
cols_with_nan  = missing_before[missing_before > 0]
if len(cols_with_nan):
    df.ffill(inplace=True)
    df.bfill(inplace=True)
    filled = cols_with_nan.sum()
    print(f"\n  Missing values")
    print(f"       Columns affected : {list(cols_with_nan.index)}")
    print(f"       Total cells filled: {filled}")
else:
    print(f"\n  Missing values     → none found, data is clean")


  Missing values     → none found, data is clean


**DUPLICATE TIMESTAMP REMOVAL**

In [9]:
dupes = df.index.duplicated().sum()
if dupes:
    df = df[~df.index.duplicated(keep="first")]
    print(f"\n  Duplicate rows     → removed {dupes} duplicates")
else:
    print(f"  Duplicate rows     → none found")

  Duplicate rows     → none found


**OUTLIER REMOVAL**

In [10]:
# IQR on log returns
before = len(df)
lr   = df["LogRet"]
q1, q3 = lr.quantile(0.25), lr.quantile(0.75)
iqr  = q3 - q1
mask = (lr >= q1 - 4*iqr) & (lr <= q3 + 4*iqr)
df   = df[mask]
removed = before - len(df)
print(f"\n Outlier removal    → removed {removed} rows "
      f"(log-return beyond 4×IQR)")
print(f"       IQR range kept    : [{q1 - 4*iqr:.6f},  {q3 + 4*iqr:.6f}]")
print(f"       Rows remaining    : {len(df):,}")


  [3c] Outlier removal    → removed 64 rows (log-return beyond 4×IQR)
       IQR range kept    : [-0.003769,  0.003789]
       Rows remaining    : 11,742


**Data SPLIT + SCALE + SEQUENCES**

In [11]:
n = len(df)
nt, nv = int(n * TRAIN_FRAC), int(n * VAL_FRAC)

sc = MinMaxScaler()
sc.fit(df.iloc[:nt])
scaled = pd.DataFrame(sc.transform(df), columns=cols, index=df.index)

def make_seqs(part):
    v = part.values; cl = part["Close"].values
    X, y = [], []
    for i in range(WINDOW, len(v)):
        X.append(v[i - WINDOW:i])
        y.append(cl[i] if TARGET == "close" else (1 if cl[i] > cl[i-1] else 0))
    return np.array(X, np.float32), np.array(y, np.float32)

Xtr, ytr = make_seqs(scaled.iloc[:nt])
Xvl, yvl = make_seqs(scaled.iloc[nt:nt + nv])
Xte, yte = make_seqs(scaled.iloc[nt + nv:])
print(f"Train {Xtr.shape} | Val {Xvl.shape} | Test {Xte.shape}")

Train (8159, 60, 19) | Val (1701, 60, 19) | Test (1702, 60, 19)


**HELPERS (models reuse them)**

In [12]:
def inv(v):
    tmp = np.zeros((len(v), len(cols))); tmp[:, close_idx] = v
    return sc.inverse_transform(tmp)[:, close_idx]

class Attention(layers.Layer):
    def build(self, shape):
        d = shape[-1]
        self.W = self.add_weight("W", (d, d), initializer="glorot_uniform")
        self.v = self.add_weight("v", (d, 1), initializer="glorot_uniform")
        super().build(shape)
    def call(self, h):
        score  = tf.nn.tanh(tf.tensordot(h, self.W, [[2], [0]]))
        weight = tf.nn.softmax(tf.tensordot(score, self.v, [[2], [0]]), axis=1)
        return tf.reduce_sum(weight * h, axis=1)

out   = lambda x: layers.Dense(1)(x) if TARGET=="close" else layers.Dense(1, activation="sigmoid")(x)
loss  = "mse" if TARGET == "close" else "binary_crossentropy"
sh    = (Xtr.shape[1], Xtr.shape[2])
cbs   = lambda: [EarlyStopping("val_loss", patience=15, restore_best_weights=True),
                 ReduceLROnPlateau("val_loss", factor=0.5, patience=7, min_lr=1e-6)]

**Models**

In [13]:
i = Input(sh)
x = layers.LSTM(UNITS, return_sequences=True)(i)
x = layers.Dropout(DROPOUT)(x)
x = layers.LSTM(UNITS)(x)
x = layers.Dropout(DROPOUT)(x)
lstm = Model(i, out(x), name="LSTM")
lstm.compile("adam", loss)

i = Input(sh)
x = layers.Bidirectional(layers.LSTM(UNITS, return_sequences=True))(i);
x = layers.Dropout(DROPOUT)(x)
x = layers.Bidirectional(layers.LSTM(UNITS // 2))(x)
x = layers.Dropout(DROPOUT)(x)
bilstm = Model(i, out(x), name="BiLSTM")
bilstm.compile("adam", loss)

i = Input(sh)
x = layers.GRU(UNITS, return_sequences=True)(i)
x = layers.Dropout(DROPOUT)(x)
x = layers.GRU(UNITS)(x)
x = layers.Dropout(DROPOUT)(x)
gru = Model(i, out(x), name="GRU")
gru.compile("adam", loss)

i = Input(sh)
x = layers.Conv1D(64, 3, activation="relu", padding="same")(i)
x = layers.MaxPooling1D(2)(x)
x = layers.Dropout(DROPOUT)(x)
x = layers.Bidirectional(layers.LSTM(UNITS, return_sequences=True))(x)
x = layers.Dropout(DROPOUT)(x)
x = Attention()(x)
x = layers.Dense(32, activation="relu")(x)
cnn_attn = Model(i, out(x), name="CNN_BiLSTM_Attn")
cnn_attn.compile("adam", loss)

MODELS = [lstm, bilstm, gru, cnn_attn]
COLORS = ["#2563EB", "#7C3AED", "#059669", "#D97706"]

TypeError: Layer.add_weight() got multiple values for argument 'initializer'